# 13 — GoalScope qualitative demo (paper-style callouts)

Interactive version of `Overleaf-draft/REF/qualitative_demo.png`: source code on the right, italic-quote read-out callouts on the left joined by leader lines to pink anchor dots.

**Interaction:** click a highlighted (scoped) token to pin its read-out — a numbered callout appears, the token is boxed + dotted, and a curve is drawn. Click again (or the ✕) to unpin. Multiple tokens can be pinned. Hover shows a transient preview.

Renderer: `src/goals/goalscope_viz.py`. Input = the `list[dict]` returned by `apply_goalscope(..., include_noncode=True, show=False)`.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
# resolve to the repo root so relative paths (head_sets/, notebooks/diff/visualizations/) work
# regardless of the notebook's launch cwd
while not os.path.isdir('src') and os.getcwd() != '/':
    os.chdir('..')
sys.path.append('.')
print('cwd:', os.getcwd())
from src.goals.goalscope_viz import goalscope_qualitative_html, show_goalscope_qualitative

cwd: /disk/u/arnab/Codes/Projects/goals


## GoalScope on Qwen2.5-Coder-32B
Load the model + the 122 goal heads, run `apply_goalscope` over a code prompt, and render. *(Heavy — needs a free GPU. Set `CUDA`.)*

In [2]:
import os, torch
MODEL = 'Qwen/Qwen2.5-Coder-32B-Instruct'
CUDA  = '0,1'   # set to free GPU(s)
os.environ['CUDA_VISIBLE_DEVICES'] = CUDA
from src.models import ModelandTokenizer
from src.goals.goalscope import apply_goalscope
from scripts.evaluation.eval_goalscope import load_json_heads

slug = MODEL.split('/')[-1]
HEADS_JSON = f'head_sets/{slug}/goal__lam20_thr0.9_band25-56.json'  # 122 goal heads
mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map='auto',
                       attn_implementation='eager')
heads = load_json_heads(HEADS_JSON, mt.n_layer, mt.model.config.num_attention_heads)
print(f'{len(heads)} goal heads loaded')

Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

122 goal heads loaded


In [6]:
from src.functional import generate_with_patch
from rich import print as rich_print

# task = "stable marriage"
task = "merge-sort"

msgs = [
    {'role': 'system', 'content': 'You are a helpful assistant proficient in programming. '
        'Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.'},
    {'role': 'user', 'content': f'Implement the {task} algorithm in python'},
    {'role': 'assistant', 'content': '<code>\n'},
]
prompt = mt.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False,
                                             continue_final_message=True)

gen = generate_with_patch(
    mt=mt,
    inputs=prompt,
    n_gen_per_prompt=1,
    do_sample=False,
    remove_prefix=True,
    max_new_tokens=256
)[0]
rich_print(f"[white]{prompt}[/white][sea_green3]{gen}[/sea_green3]")

<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the 
code, no need to comment or explain.<|im_end|>
<|im_start|>user
Implement the merge-sort algorithm in python<|im_end|>
<|im_start|>assistant
<code>
def merge_sort(arr):
    if len(arr) > 1:
        mid = len(arr) // 2
        L = arr[:mid]
        R = arr

        merge_sort(L)
        merge_sort(R)

        i = j = k = 0

        while i < len(L) and j < len(R):
            if L < R:
                arr = L
                i += 1
            else:
                arr = R
                j += 1
            k += 1

        while i < len(L):
            arr = L
            i += 1
            k += 1

        while j < len(R):
            arr = R
            j += 1
            k += 1
</code>

In [7]:
# Verbalize every code token (include_noncode=True keeps the whole prompt; non-code tokens dimmed).
res = apply_goalscope(prompt+gen, mt=mt, selected_heads=heads, strategy='additive_norm',
                      steer_layer_range=(35, 46), alpha=0.1, n_steer_tokens='all',
                      max_new_tokens=64, include_noncode=True, show=False)
print(sum(1 for r in res if r.get('scoped')), 'scoped tokens')

GoalScope[additive_norm]:   0%|          | 0/12 [00:00<?, ?it/s]

177 scoped tokens


In [ ]:
# Inline (interactive) + save a standalone .html for the paper.
import pathlib; pathlib.Path('notebooks/diff/visualizations').mkdir(parents=True, exist_ok=True)
goalscope_qualitative_html(res, title=f'{task} — GoalScope',
                           save_path=f'notebooks/diff/visualizations/{task}.html')
show_goalscope_qualitative(res, title=f'{task} — GoalScope', height=1224)

: 